## **FLOPS**

`(Floating Point Operations)` or `FLOPs` are a measure of computational cost, essentially counting the number of mathematical operations involving decimal numbers performed by a model. They are crucial for estimating training time, inference speed, and energy consumption. In the context of transformers, both self-attention and dense layers are computationally intensive, contributing significantly to the overall FLOPs.

**Attention Layer**

In a transformer model, attention layers enable the model to `weigh` the importance of `different parts` of the input sequence when processing a specific element, thereby understanding context. This mechanism, often using `Query`, `Key`, and `Value` vectors, allows the model to capture long-range dependencies in data, which is vital for language understanding.

**MLP / Dense Layer**

Dense layers, on the other hand, are fully connected layers that perform `linear transformations` followed by `non-linear` activations. They help in learning complex patterns by combining features extracted from previous layers. Both `attention` and `dense layers` require significant computational resources, leading to high FLOPs, especially in large-scale models.

<hr>

### **Effect of `FLOPS` as Model Scales**

For smaller models, the `FLOPs` spent in `attention layers` and MLP layers are roughly comparable i.e. similar. However, as the models scale up, for instance, to `175` billion parameters, the `MLP` layers begin to strongly dominate the `FLOPs` consumption.

So as the model size increases, the proportion of `FLOPs` used by `MLP layers` becomes significantly larger compared to that used by `attention layers`. This shift indicates that in very large models, the computational burden is increasingly borne by the dense layers rather than the attention mechanisms.

Therefore, 

This observation is critical because it indicates that optimizing different parts of the model might be more beneficial at different scales. 

If researchers primarily work with small models and spend a lot of time optimizing the attention mechanism, they might be `"optimizing the wrong thing"` for larger-scale models. At a larger scale, where MLPs consume a significantly higher fraction of FLOPs, optimizations in the MLP architecture or operations would likely yield much greater efficiency gains.

<hr>
<hr>
<hr>


## **Improvements in `Transformer` Architectures**

> `Transformer` by default uses `Dense` Feed Forward Networks (FFN) in its architecture. However, a different architecture, `Mixture of Experts (MoE)`, has been proposed as an alternative to the dense FFN.

We'll try to understand the differences between these two architectures and their implications on model performance and efficiency.

<hr>

### **Dense Feed Forward Networks (FFN)**

In a `Dense FFN`, every input is processed through the same set of parameters. This means that all inputs are treated equally, and the model learns a single representation for all data points. While this can be effective for smaller models, it can lead to inefficiencies as the model scales up, especially when different inputs might require different processing.

Mathematically, 

$$
\text{Output} = \sigma(\text{Input} \cdot W_1 + b_1) \cdot W_2 + b_2
$$

Where,

- $\sigma$ is the activation function (e.g., ReLU, SwiGLU)

- $W_1$ and $W_2$ are weight matrices

Let's understand better with an `Analogy`. `Dense` FFN is like a Single Teacher teaching:

- Maths Students

- Literature Students

- Physics Students

Here, the teacher (model) has to cater to the needs of all students (inputs) with the same teaching method (parameters), which might not be optimal for every student.

> It works, but it's inefficient and suboptimal for larger models where different inputs might benefit from different processing.

**Limitations of Dense FFN**

- Compute scales Linearly with the number of parameters, which can lead to inefficiencies in larger models. As the model size increases -> More `FLOPs` are required, leading to longer training times and higher energy consumption.

- It may not capture the diversity of inputs effectively, as all inputs are processed through the same parameters, which can limit the model's ability to learn complex patterns.

  - Same neurons handle: `Code`, `Maths`, `Literature` etc.

<hr>

### **Mixture of Experts (MoE)**

> What if only part of a the network is activated for a given input, allowing the model to specialize and potentially reduce computational costs? Instead of using all neurons for every input.

In contrast, a `Mixture of Experts (MoE)` architecture consists of multiple "expert" sub-networks, each specializing in different aspects of the input data. A gating mechanism is used to determine which expert(s) should be activated for a given input, allowing the model to dynamically route inputs to the most relevant experts.

`Mixture of Experts (MoE)` replaces the `Dense` FFN with:

- Many independent sub-networks `Experts` that specialize in different aspects of the input data

- A `Router Gate` that decides which expert(s) to activate for each input

```bash

Input token
   ↓
 Router (Gate)
   ↓
 Selected Experts (Top-k)
   ↓
 Weighted Sum
   ↓
 Output

```

But, 

**What is an `Expert`?**

An `Expert` is a sub-network within the `MoE` architecture that specializes in processing specific types of inputs. 

It is just a small `Feed Forward Network (FFN)` that is trained to handle a particular subset of the input data. Each expert learns to focus on different features or patterns in the data, allowing the model to capture a wider range of information.

For example, in a language model, one expert might specialize in understanding `code`, while another might focus on `literature`. When an input token is processed, the router gate determines which expert(s) are most relevant for that token and activates them accordingly.

And,

**What is a `Router Gate`?**

A `Router Gate` is a mechanism within the `MoE` architecture that decides which expert(s) to activate for a given input. It takes the input token and computes a score for each expert, determining their relevance to the input.

It is a small neural network computes a score for each expert based on the input token, and then selects the top-k experts with the highest scores to process the input. The output from the selected experts is then combined (e.g., through a weighted sum) to produce the final output.

- It decides which expert(s) to activate for each input token

- Usually Top-k experts are selected based on the scores computed by the router gate

<hr>

> `MoE` increases model capacity without increasing compute by activating only a subset of experts for each input, allowing the model to learn more complex patterns while keeping computational costs manageable.

> For example, if we've 64 experts but only activate 4 for each input, we can have a much larger model capacity without a proportional increase in FLOPs.

<hr>

### **How does `Mixture of Experts (MoE)` work?**

> Important: `Routing` happens per token, not per sentence. Each token in the input sequence can be routed to different experts based on its characteristics.

For example, the input is `"The derivative of sin(x) is cos(x)"`. Then, 

| Token      | Routed Expert  |
| ---------- | -------------- |
| derivative | math expert    |
| sin        | math expert    |
| is         | general expert |
| cos        | math expert    |

As `Language` is inherently multi-domain, tokens can come from various domains (e.g., code, literature, math), and `MoE` allows the model to route each token to the most relevant expert, improving efficiency and performance.

<hr>

**`MoE` w `Transformers`**

```bash

Self-Attention
↓
Normalization
↓
MoE FFN  ← replaces Dense FFN
↓
Residual
```

**Famous `MoE` Models**

- `GShard` (Google): One of the earliest implementations of MoE in transformers, demonstrating significant improvements in model capacity and performance.

- `Switch Transformer` (Google): A more efficient MoE model that uses a simpler routing mechanism, achieving state-of-the-art results with fewer parameters.